# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

# Completed solution: local music recommendations

This solution uses **eight real music recordings** from [librosa's example collection](https://librosa.org/doc/main/recordings.html), an alternative online dataset allowed by the assignment. The catalog includes classical, jazz, electronic, folk, waltz, and ragtime. Source and licensing notices are retained in `data/attributions/`; genre and mood tags are manually curated for this lab.

**Pipeline:** music files ? 53 audio features + catalog text ? persistent local vector index ? preference/seed retrieval ? grounded recommendation text and playback.

Generation uses a deterministic template populated from retrieved metadata. This is a lightweight retrieval-augmented generation pipeline, **not an LLM**. TF-IDF handles catalog vocabulary; it does not understand arbitrary language or negation. The original cloud tutorial is retained below as an optional section.

## 1. Setup

Use Python 3.10+ and run `python -m pip install -r requirements.txt` in the project directory. Select that Python environment as the notebook kernel. All music files are included; if missing, run `python scripts/download_music.py` once. Index building and querying then work offline, without API keys.

In [1]:
from pathlib import Path
import json
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from music_recommendations import (ROOT, DEFAULT_CATALOG, DEFAULT_INDEX,
    EMBEDDING_DIM, LocalMusicRAG, build_index, extract_embedding)

catalog = json.loads(DEFAULT_CATALOG.read_text(encoding="utf-8"))
missing = [track["path"] for track in catalog if not (ROOT / "data" / track["path"]).is_file()]
assert not missing, f"Run python scripts/download_music.py first: {missing}"
print(f"Music catalog: {len(catalog)} distinct recordings")
for track in catalog:
    print(f"{track['id']:12} | {track['genre']:10} | {track['title']} ? {track['artist']}")

Music catalog: 8 distinct recordings
brahms       | classical  | Hungarian Dance No. 5 ? Johannes Brahms / US Army Strings
nutcracker   | classical  | Dance of the Sugar Plum Fairy ? Pyotr Ilyich Tchaikovsky / Kevin MacLeod
choice       | electronic | Choice ? Admiral Bob
fishin       | folk       | Let?s Go Fishin? ? Karissa Hobbs
sweetwaltz   | waltz      | Sweet Waltz ? Setuniman
pistachio    | ragtime    | Pistachio Ice Cream Ragtime ? The Piano Lady
trumpet      | jazz       | Trumpet loop ? Mihai Sorohan
vibeace      | jazz       | Vibe Ace ? Kevin MacLeod


## 2. Build and persist the local index

Read at most 30 seconds per track; mix to mono and resample to 22,050 Hz. Features are 20 MFCC means, 20 MFCC variances, 12 chroma means, and one RMS value. Each feature is standardized using **catalog-only** statistics, then each vector is L2-normalized. Text documents combine title, artist, description, genre, and tags into TF-IDF vectors.

Audio similarity is cosine similarity. Text similarity is TF-IDF cosine. The `.npz` index stores audio vectors, scaling statistics, and metadata, with no pickle objects. A content fingerprint causes rebuilding when audio, catalog, or feature version changes. An unchanged index is reused.

In [2]:
rag = build_index()
print("Audio index:", rag.vectors.shape)
print("Text index:", rag.text_vectors.shape)
assert rag.vectors.shape == (len(catalog), EMBEDDING_DIM)
assert np.isfinite(rag.vectors).all()
assert np.allclose(np.linalg.norm(rag.vectors, axis=1), 1)
print("Saved locally:", DEFAULT_INDEX.name)
loaded = LocalMusicRAG.load()
assert np.allclose(loaded.vectors, rag.vectors)
print("Reload verified")

Audio index: (8, 53)
Text index: (8, 81)
Saved locally: music_index.npz
Reload verified


## 3. Recommend from user preferences

Edit `preferences` to express positive preferences using words in the catalog, such as piano, jazz, vocals, guitar, classical, orchestra, or electronic. Use `genre` for an exact genre filter and `exclude_ids` for explicit exclusions. Text-only search returns only tracks with overlapping terms, so fewer than `top_k` results can be returned. Unknown terms produce a clear error instead of arbitrary recommendations.

In [3]:
preferences = "relaxed jazz piano"
results = rag.retrieve(preferences=preferences, top_k=3)
print(rag.generate_answer(results))

1. Vibe Ace — Kevin MacLeod [vibeace]
   A vibraphone, piano, and bass combo. Catalog terms: jazz, piano, relaxed.
   Retrieval score: 0.512; source: https://librosa.org/data/audio/Kevin_MacLeod_-_Vibe_Ace.txt
2. Pistachio Ice Cream Ragtime — The Piano Lady [pistachio]
   A solo piano ragtime piece. Catalog terms: piano.
   Retrieval score: 0.236; source: https://librosa.org/data/audio/442789__lena-orsa__happy-music-pistachio-ice-cream-ragtime.txt
3. Trumpet loop — Mihai Sorohan [trumpet]
   A monophonic trumpet loop from a jazz trumpet loop pack. Catalog terms: jazz.
   Retrieval score: 0.178; source: https://librosa.org/data/audio/sorohanro_-_solo-trumpet-06.txt


## 4. Combine preferences with favorite tracks

Select one or more catalog IDs as favorites. Their audio vectors are averaged and normalized. The hybrid score is `audio_weight * (audio_cosine + 1)/2 + (1-audio_weight) * text_cosine`. For a single modality, only that modality is used. Scores rank candidates; they are not probabilities or calibrated relevance estimates. Favorite tracks are excluded from results.

In [4]:
hybrid = rag.retrieve(preferences="jazz piano", seed_ids=["trumpet"],
                      audio_weight=0.5, top_k=3)
assert "trumpet" not in [track["id"] for track in hybrid]
print(rag.generate_answer(hybrid))

1. Vibe Ace — Kevin MacLeod [vibeace]
   A vibraphone, piano, and bass combo. Catalog terms: jazz, piano.
   Retrieval score: 0.377; source: https://librosa.org/data/audio/Kevin_MacLeod_-_Vibe_Ace.txt
2. Pistachio Ice Cream Ragtime — The Piano Lady [pistachio]
   A solo piano ragtime piece. Catalog terms: piano.
   Retrieval score: 0.334; source: https://librosa.org/data/audio/442789__lena-orsa__happy-music-pistachio-ice-cream-ragtime.txt
3. Sweet Waltz — Setuniman [sweetwaltz]
   A waltz in 3/4 with synthesized xylophone, flute, and organ. Retrieved by seed/audio similarity.
   Retrieval score: 0.331; source: https://librosa.org/data/audio/147793__setuniman__sweet-waltz-0i-22mi.txt


## 5. Listen to the recommendations

The generated answer uses only retrieved descriptions and matching catalog terms, cites track IDs and source URLs, and makes no claim that a language model listened to the audio. Listen to these excerpts to judge subjective relevance.

In [5]:
for track in hybrid[:2]:
    print(f"{track['title']} [{track['id']}]")
    with sf.SoundFile(ROOT / "data" / track["path"]) as audio:
        excerpt = audio.read(frames=audio.samplerate * 5, always_2d=True).mean(axis=1)
        display(Audio(excerpt, rate=audio.samplerate))

Vibe Ace [vibeace]


Pistachio Ice Cream Ragtime [pistachio]


## 6. Query a recording held out of the index

This demonstration leaves the trumpet recording out of both audio standardization and text indexing, then uses its file as the external query. It checks the retrieval path for previously unseen audio; there is no claim that the returned tracks are ground-truth matches.

In [6]:
import tempfile

with tempfile.TemporaryDirectory() as temporary:
    heldout_catalog = []
    for track in catalog:
        if track["id"] != "trumpet":
            heldout_catalog.append({**track, "path": str(ROOT / "data" / track["path"])})
    manifest = Path(temporary) / "catalog.json"
    manifest.write_text(json.dumps(heldout_catalog), encoding="utf-8")
    heldout_rag = build_index(manifest, Path(temporary) / "index.npz")
    external = heldout_rag.retrieve(query_audio=ROOT / "data/music/trumpet.ogg", top_k=3)
    assert len(heldout_rag.tracks) == 7
    assert all(track["id"] != "trumpet" for track in external)
    print(heldout_rag.generate_answer(external))

1. Sweet Waltz — Setuniman [sweetwaltz]
   A waltz in 3/4 with synthesized xylophone, flute, and organ. Retrieved by seed/audio similarity.
   Retrieval score: 0.674; source: https://librosa.org/data/audio/147793__setuniman__sweet-waltz-0i-22mi.txt
2. Dance of the Sugar Plum Fairy — Pyotr Ilyich Tchaikovsky / Kevin MacLeod [nutcracker]
   An orchestral excerpt from the Nutcracker with a delicate melody. Retrieved by seed/audio similarity.
   Retrieval score: 0.508; source: https://librosa.org/data/audio/Kevin_MacLeod_-_P_I_Tchaikovsky_Dance_of_the_Sugar_Plum_Fairy.txt
3. Choice — Admiral Bob [choice]
   A short drum and bass loop. Retrieved by seed/audio similarity.
   Retrieval score: 0.506; source: https://librosa.org/data/audio/admiralbob77_-_Choice_-_Drum-bass.txt


## 7. Sanity evaluation and limitations

The cases below test whether explicit catalog preferences retrieve their intended tracks at rank 1. These are **handwritten smoke checks against curated tags**, not an independent benchmark or evidence of general recommendation quality. The held-out demonstration above separately checks unseen audio input. Automated tests also cover empty/silent files, dimension consistency, persistence, invalid requests, and query exclusion.

In [7]:
cases = [
    ("solo piano ragtime", "pistachio"),
    ("energetic drum bass electronic", "choice"),
    ("folk guitar vocals", "fishin"),
    ("classical orchestra strings", "brahms"),
    ("relaxed jazz piano", "vibeace"),
]
hits = 0
for preference, expected in cases:
    actual = rag.retrieve(preferences=preference, top_k=1)[0]["id"]
    hit = actual == expected
    hits += hit
    print(f"{preference:34} expected={expected:10} retrieved={actual:10} pass={hit}")
print(f"Catalog smoke checks: {hits}/{len(cases)}")
assert hits == len(cases)

filtered = rag.retrieve("classical orchestra", genre="classical", top_k=5)
assert filtered and all(track["genre"] == "classical" for track in filtered)
assert rag.retrieve("piano", genre="unknown") == []
try:
    rag.retrieve("zxqv")
except ValueError as error:
    print("Unsupported preference:", error)
else:
    raise AssertionError("Unknown preferences must not return arbitrary tracks")

solo piano ragtime                 expected=pistachio  retrieved=pistachio  pass=True
energetic drum bass electronic     expected=choice     retrieved=choice     pass=True
folk guitar vocals                 expected=fishin     retrieved=fishin     pass=True
classical orchestra strings        expected=brahms     retrieved=brahms     pass=True
relaxed jazz piano                 expected=vibeace    retrieved=vibeace    pass=True
Catalog smoke checks: 5/5
Unsupported preference: No recognized preferences; try piano, jazz, classical, vocals, or a seed ID


### Discussion

The system meets the local workflow: real music ingestion, reusable audio/text representations, preference and seed queries, retrieval, grounded answer generation, source attribution, and playback. It supports adding licensed recordings through `data/catalog.json` and rebuilding the index.

Limitations: eight tracks are too few for broad discovery; the first 30 seconds may not represent an entire song; handcrafted features capture timbre/harmony better than semantics; audio similarity does not imply shared genre; curated tags introduce bias; TF-IDF does not interpret negation or synonyms. Metadata smoke checks use the same tags being searched and therefore overestimate broader relevance. Improvements would include a larger licensed corpus, multiple excerpts, learned music/text embeddings, independent relevance judgments, and diversity-aware ranking.

# Optional: original Pinecone environmental-audio tutorial

The supplied tutorial is preserved for comparison with the local solution. Its code is disabled by default so **Run All** works offline and cannot delete a cloud index. This section has not been validated against a live Pinecone account. To explore it, install its optional packages and provide your own `PINECONE_API_KEY`, then set `RUN_PINECONE_DEMO=True`. PANNs input has been adjusted to 32 kHz, and vector queries now pass one flat embedding. Index deletion requires the separate flag as well.

In [8]:
RUN_PINECONE_DEMO = False
DELETE_PINECONE_INDEX = False
# Optional dependencies (only for the original tutorial):
# %pip install librosa panns-inference datasets pinecone python-dotenv tqdm torch

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [9]:
if RUN_PINECONE_DEMO:
    #!pip install librosa
    #!pip install panns-inference
    pass

In [10]:
if RUN_PINECONE_DEMO:
    # !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa
    pass

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [11]:
if RUN_PINECONE_DEMO:
    from datasets import load_dataset
    
    # load the dataset from huggingface model hub
    data = load_dataset("ashraq/esc50", split="train")
    data
    from datasets import Audio as DatasetAudio
    data = data.cast_column("audio", DatasetAudio(sampling_rate=32000))
    pass

The audios in the dataset are sampled at 32000Hz (resampled for PANNs) and loaded into NumPy arrays. Let's take a look.

In [12]:
if RUN_PINECONE_DEMO:
    # select the audio feature and display top three
    audios = data["audio"]
    audios[:3]
    pass

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [13]:
if RUN_PINECONE_DEMO:
    import numpy as np
    
    # select only the audio data from the dataset and store in a numpy array
    audios = np.array([a["array"] for a in data["audio"]])
    pass

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [14]:
if RUN_PINECONE_DEMO:
    from panns_inference import AudioTagging
    import torch
    
    # Use CUDA when available; otherwise fall back to CPU so the notebook works on Windows and other environments.
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AudioTagging(checkpoint_path=None, device=device)
    pass

## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [15]:
if RUN_PINECONE_DEMO:
    from dotenv import load_dotenv, find_dotenv
    import os
    _ = load_dotenv(find_dotenv())
    
    OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
    # initialize connection to pinecone (get API key at app.pinecone.io)
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    if not PINECONE_API_KEY:
        raise ValueError("Set the PINECONE_API_KEY environment variable before running the Pinecone indexing cells.")
    pass

In [16]:
if RUN_PINECONE_DEMO:
    import os
    from pinecone import Pinecone
    
    # configure client
    pc = Pinecone(api_key=PINECONE_API_KEY)
    pass

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [17]:
if RUN_PINECONE_DEMO:
    from pinecone import ServerlessSpec
    
    cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
    region = os.environ.get('PINECONE_REGION') or 'us-east-1'
    
    spec = ServerlessSpec(cloud=cloud, region=region)
    pass

Create the index:

In [18]:
if RUN_PINECONE_DEMO:
    index_name = "audio-search-demo"
    pass

In [19]:
if RUN_PINECONE_DEMO:
    import time
    
    # check if index already exists (it shouldn't if this is first time)
    if index_name not in pc.list_indexes().names():
        # if does not exist, create index
        pc.create_index(
            index_name,
            dimension=2048,
            metric='cosine',
            spec=spec
        )
        # wait for index to be initialized
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)
    
    # connect to index
    index = pc.Index(index_name)
    # view index stats
    index.describe_index_stats()
    pass

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [20]:
if RUN_PINECONE_DEMO:
    from tqdm.auto import tqdm
    
    # we will use batches of 64
    batch_size = 64
    
    for i in tqdm(range(0, len(audios), batch_size)):
        # find end of batch
        i_end = min(i+batch_size, len(audios))
        # extract batch
        batch = audios[i:i_end]
        # generate embeddings for all the audios in the batch
        _, emb = model.inference(batch)
        # create unique IDs
        ids = [f"{idx}" for idx in range(i, i_end)]
        # add all to upsert list
        to_upsert = list(zip(ids, emb.tolist()))
        # upsert/insert these records to pinecone
        _ = index.upsert(vectors=to_upsert)
    
    # check that we have all vectors in index
    index.describe_index_stats()
    pass

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [21]:
if RUN_PINECONE_DEMO:
    from IPython.display import Audio, display
    
    # we set an audio number to select from the dataset
    audio_num = 400
    # get the audio data of the audio number
    query_audio = data[audio_num]["audio"]["array"]
    # get the category of the audio number
    category = data[audio_num]["category"]
    # print the category and play the audio
    print("Query Audio:", category)
    Audio(query_audio, rate=32000)
    pass

We have got the sound of a car horn. Let's generate an embedding for this sound.

In [22]:
if RUN_PINECONE_DEMO:
    # reshape query audio
    query_audio = query_audio[None, :]
    # get the embeddings for the audio from the model
    _, xq = model.inference(query_audio)
    xq.shape
    pass

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [23]:
if RUN_PINECONE_DEMO:
    # query pinecone index with the query audio embeddings
    results = index.query(vector=xq[0].tolist(), top_k=3)
    results
    pass

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [24]:
if RUN_PINECONE_DEMO:
    # play the top 3 similar audios
    for r in results["matches"]:
        # select the audio data from the databse using the id as an index
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=32000))
    pass

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [25]:
if RUN_PINECONE_DEMO:
    def find_similar_audios(id):
        print("Query Audio:")
        # select the audio data from the databse using the id as an index
        query_audio = data[id]["audio"]["array"]
        # play the query audio
        display(Audio(query_audio, rate=32000))
        # query pinecone index with the query audio id
        result = index.query(id=str(id), top_k=5)
        print("Result:")
        # play the top 5 similar audios
        for r in result["matches"]:
            a = data[int(r["id"])]["audio"]["array"]
            display(Audio(a, rate=32000))
    pass

In [26]:
if RUN_PINECONE_DEMO:
    find_similar_audios(1642)
    pass

Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [27]:
if RUN_PINECONE_DEMO:
    find_similar_audios(452)
    pass

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [28]:
if RUN_PINECONE_DEMO:
    #!wget https://storage.googleapis.com/audioset/miaow_16k.wav
    pass

We can load the audio into a Numpy array as follows:

In [29]:
if RUN_PINECONE_DEMO:
    import librosa
    
    a, _ = librosa.load("./data/miaow_16k.wav", sr=32000)
    Audio(a, rate=32000)
    pass

Now we generate the embeddings for this audio and query the Pinecone index.

In [30]:
if RUN_PINECONE_DEMO:
    # reshape query audio
    query_audio = a[None, :]
    # get the embeddings for the audio from the model
    _, xq = model.inference(query_audio)
    
    # query pinecone index with the query audio embeddings
    results = index.query(vector=xq[0].tolist(), top_k=3)
    
    # play the top 3 similar audios
    for r in results["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=32000))
    pass

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [31]:
if RUN_PINECONE_DEMO and DELETE_PINECONE_INDEX:
    pc.delete_index(index_name)
    pass